<a href="https://colab.research.google.com/github/melissaboukerchi55-create/principe_de_programmation/blob/main/Projet_Allocine_essentiel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projet Allociné : l'essentiel

**Le but :** lire un avis de film en français et dire s'il est **positif** ou **négatif**.

**Avant de lancer :**
1. Menu **Exécution** → **Modifier le type d'exécution** → choisis **GPU (T4)** → **Enregistrer**.
2. Menu **Exécution** → **Tout exécuter**. Ça prend environ 20 à 30 minutes.

## 1. Les données : 200 000 avis Allociné

In [14]:
import os, tarfile, urllib.request
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/TheophileBlard/french-sentiment-analysis-with-bert/master/allocine_dataset/data.tar.bz2"
if not os.path.exists("data/train.jsonl"):
    urllib.request.urlretrieve(url, "allocine.tar.bz2")
    tarfile.open("allocine.tar.bz2").extractall()

app = pd.read_json("data/train.jsonl", lines=True)    # pour apprendre
val = pd.read_json("data/val.jsonl", lines=True)      # pour régler
test = pd.read_json("data/test.jsonl", lines=True)    # pour la note finale

print("Apprentissage :", len(app), "| Validation :", len(val), "| Test :", len(test))
print("Positifs / négatifs :", (app.polarity == 1).sum(), "/", (app.polarity == 0).sum())
print()
for i in range(2):
    print("Étiquette", app.polarity[i], ":", app.review[i][:150])

Apprentissage : 160000 | Validation : 20000 | Test : 20000
Positifs / négatifs : 80587 / 79413

Étiquette 0 : Si vous cherchez du cinéma abrutissant à tous les étages,n'ayant aucune peur du cliché en castagnettes et moralement douteux,"From Paris with love" es
Étiquette 0 : Trash, re-trash et re-re-trash...! Une horreur sans nom. Imaginez-vous les 20 premières minutes de Orange Mécanique dilatées sur plus de 70 minutes de


## 2. Le modèle simple : TF-IDF + régression logistique
- **TF-IDF** transforme chaque avis en nombres (un nombre par mot).
- **La régression logistique** apprend quels mots rendent un avis positif ou négatif.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=300000, sublinear_tf=True)
X_app = tfidf.fit_transform(app.review)
X_test = tfidf.transform(test.review)

logreg = LogisticRegression(C=4, max_iter=2000, solver="liblinear")
logreg.fit(X_app, app.polarity)

pred_simple = logreg.predict(X_test)
print(f"Modèle simple sur le test : accuracy = {accuracy_score(test.polarity, pred_simple):.4f} | F1 = {f1_score(test.polarity, pred_simple):.4f}")

poids = pd.Series(logreg.coef_[0], index=tfidf.get_feature_names_out()).sort_values()
print("\nMots les plus NÉGATIFS :", list(poids.head(8).index))
print("Mots les plus POSITIFS :", list(poids.tail(8).index[::-1]))

Modèle simple sur le test : accuracy = 0.9452 | F1 = 0.9430

Mots les plus NÉGATIFS : ['ennuyeux', 'ennui', 'mauvais', 'navet', 'intérêt', 'déception', 'plv', 'nul']
Mots les plus POSITIFS : ['excellent', 'magnifique', 'superbe', 'bon film', 'pas mal', 'très bon', 'beau film', 'génial']


## 3. Tester des phrases (regarde la négation « pas mauvais »)

In [16]:
phrases = ["Ce film est magnifique, je le recommande à tous !",
           "Je m'attendais à mieux, quelle déception.",
           "Ce n'est pas un mauvais film."]
for phrase in phrases:
    proba = logreg.predict_proba(tfidf.transform([phrase]))[0, 1]
    print(f"{proba:6.1%} positif | {phrase}")

100.0% positif | Ce film est magnifique, je le recommande à tous !
  0.0% positif | Je m'attendais à mieux, quelle déception.
  0.0% positif | Ce n'est pas un mauvais film.


## 4. CamemBERT : le deep learning (GPU obligatoire)
CamemBERT est un **Transformer** déjà entraîné sur le français. On le réentraîne un peu sur nos avis : c'est le **fine-tuning** (transfer learning).

In [17]:
!pip install -q transformers accelerate sentencepiece

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

print("GPU disponible :", torch.cuda.is_available())     # doit afficher True

nom_modele = "camembert-base"       # si erreur, remplace par "almanach/camembert-base"
tokenizer = AutoTokenizer.from_pretrained(nom_modele)
modele = AutoModelForSequenceClassification.from_pretrained(nom_modele, num_labels=2)

app_petit = app.sample(10000, random_state=0)       # 10 000 avis pour apprendre (sinon trop long)
val_petit = val.sample(2000, random_state=0)

class JeuAvis(torch.utils.data.Dataset):
    def __init__(self, textes, etiquettes):
        self.enc = tokenizer(list(textes), truncation=True, padding="max_length", max_length=256)
        self.y = list(etiquettes)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        element = {cle: torch.tensor(v[i]) for cle, v in self.enc.items()}
        element["labels"] = torch.tensor(self.y[i])
        return element

jeu_app, jeu_val, jeu_test = JeuAvis(app_petit.review, app_petit.polarity), JeuAvis(val_petit.review, val_petit.polarity), JeuAvis(test.review, test.polarity)

def mesures(sortie):
    logits, vraies = sortie
    p = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(vraies, p), "f1": f1_score(vraies, p)}

reglages = TrainingArguments(output_dir="camembert", num_train_epochs=2, per_device_train_batch_size=16,
                             per_device_eval_batch_size=64, learning_rate=2e-5, weight_decay=0.01,
                             eval_strategy="epoch", save_strategy="no", logging_steps=200,
                             fp16=torch.cuda.is_available(), report_to="none")
entraineur = Trainer(model=modele, args=reglages, train_dataset=jeu_app, eval_dataset=jeu_val, compute_metrics=mesures)
entraineur.train()

GPU disponible : True


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.163007,0.164316,0.952500,0.952381
2,0.121593,0.153540,0.957500,0.957006


TrainOutput(global_step=1250, training_loss=0.17040305671691894, metrics={'train_runtime': 255.1377, 'train_samples_per_second': 78.389, 'train_steps_per_second': 4.899, 'total_flos': 2631110553600000.0, 'train_loss': 0.17040305671691894, 'epoch': 2.0})

In [19]:
resultat = entraineur.predict(jeu_test)
pred_camembert = np.argmax(resultat.predictions, axis=1)
print(f"CamemBERT sur le test : accuracy = {accuracy_score(test.polarity, pred_camembert):.4f} | F1 = {f1_score(test.polarity, pred_camembert):.4f}")
print("✍️ Note ce chiffre : c'est TON résultat CamemBERT pour l'entretien.\n")

modele.eval()
enc = tokenizer(phrases, truncation=True, padding=True, max_length=256, return_tensors="pt").to(modele.device)
with torch.no_grad():
    probas = torch.softmax(modele(**enc).logits, dim=1)[:, 1].cpu().numpy()
for phrase, proba in zip(phrases, probas):
    print(f"{proba:6.1%} positif | {phrase}")

CamemBERT sur le test : accuracy = 0.9587 | F1 = 0.9574
✍️ Note ce chiffre : c'est TON résultat CamemBERT pour l'entretien.

 99.1% positif | Ce film est magnifique, je le recommande à tous !
  0.6% positif | Je m'attendais à mieux, quelle déception.
  5.2% positif | Ce n'est pas un mauvais film.


## 5. La comparaison

In [20]:
lignes = [{"modèle": "TF-IDF + régression logistique", "accuracy": accuracy_score(test.polarity, pred_simple), "F1": f1_score(test.polarity, pred_simple)}]
if "pred_camembert" in globals():
    lignes.append({"modèle": "CamemBERT (fine-tuning)", "accuracy": accuracy_score(test.polarity, pred_camembert), "F1": f1_score(test.polarity, pred_camembert)})
print(pd.DataFrame(lignes).round(4).to_string(index=False))

                        modèle  accuracy     F1
TF-IDF + régression logistique    0.9452 0.9430
       CamemBERT (fine-tuning)    0.9587 0.9574


## 6. L'API : mettre le modèle à disposition (FastAPI)
Une **API** est un guichet : une autre application envoie un avis, et elle reçoit la réponse.

In [21]:
import joblib, sklearn
from sklearn.pipeline import Pipeline
joblib.dump(Pipeline([("tfidf", tfidf), ("logreg", logreg)]), "modele_sentiment.joblib")
print("Modèle sauvegardé.")

Modèle sauvegardé.


In [22]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

app = FastAPI(title="Sentiment des avis Allociné")
modele = joblib.load("modele_sentiment.joblib")

class Avis(BaseModel):
    texte: str

@app.post("/predire")
def predire(avis: Avis):
    proba = float(modele.predict_proba([avis.texte])[0, 1])
    return {"sentiment": "positif" if proba >= 0.5 else "négatif", "probabilite_positif": round(proba, 3)}

Overwriting app.py


In [23]:
!pip install -q fastapi httpx

In [24]:
from fastapi.testclient import TestClient
import app as api

client = TestClient(api.app)
for texte in ["Un film magnifique, à voir absolument !", "Quel navet, je me suis ennuyé."]:
    print(client.post("/predire", json={"texte": texte}).json(), "|", texte)

{'sentiment': 'positif', 'probabilite_positif': 1.0} | Un film magnifique, à voir absolument !
{'sentiment': 'négatif', 'probabilite_positif': 0.0} | Quel navet, je me suis ennuyé.


In [25]:
%%writefile Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py modele_sentiment.joblib ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


In [26]:
with open("requirements.txt", "w") as f:
    f.write(f"fastapi\nuvicorn\njoblib\nscikit-learn=={sklearn.__version__}\n")
print("Dockerfile et requirements.txt prêts. Docker sert à mettre l'API dans une « boîte » qui marche partout.")

Dockerfile et requirements.txt prêts. Docker sert à mettre l'API dans une « boîte » qui marche partout.


## À retenir pour l'entretien
- **Données :** 200 000 avis Allociné, moitié positifs, moitié négatifs.
- **Modèle simple :** TF-IDF + régression logistique → **94,5 %** de bonnes réponses. Mais il ne comprend pas « **pas mauvais** ».
- **CamemBERT :** un Transformer déjà entraîné sur le français, réentraîné sur nos avis (fine-tuning) → ✍️ **ton chiffre**. Il comprend mieux le sens de la phrase grâce à l'**attention**.
- **API FastAPI + Docker :** pour que d'autres applications puissent utiliser le modèle.
- **Lien EDF :** le fine-tuning, c'est du transfer learning. Pour les images ultrasonores, on ferait pareil avec un réseau déjà entraîné sur des images.

> « J'ai classé 200 000 avis Allociné en positifs ou négatifs. Un modèle simple, TF-IDF avec une régression logistique, fait 94,5 %, mais il ne comprend pas les négations. J'ai donc fait le fine-tuning de CamemBERT, un Transformer déjà entraîné sur le français, qui monte à ✍️ ton chiffre. Enfin, j'ai mis le modèle dans une API FastAPI, avec Docker. »